# Chain-depth calibration on Colab

Chooses the chain depths `d` and the middle budget for the main grid. The single agent runs on development problems at several depths, with every prompt holding the same number of equations, so only the dependency path changes with `d`. Useful depths keep single-agent accuracy at the middle budget away from 0% and 100%, so the other systems have room to do better or worse.

These problems use the development seed. The main grid must use a different seed, so prompts are never tuned on test problems.

The same notebook runs other development checks by changing the settings cell: `SYSTEMS` adds the ensemble and hierarchy, and `RUN_NAME` gives each run its own results folder.

Use **Runtime > Change runtime type > L4 GPU**. Results go to `MyDrive/small-multi-agents/results/<RUN_NAME>/`, with the generated problems, a `run.log`, and summary CSV files. If the session disconnects, rerun every cell. Runs resume from the last saved batch. Changing any setting needs a new `RUN_NAME`, since resume refuses logs for different problems or settings.

In [ ]:
RUN_NAME = "calibrate_depth"   # results folder; use a new name for any other settings
SYSTEMS = ["single_agent"]     # any of "single_agent", "ensemble", "hierarchy"
DEPTHS = [1, 2, 4, 6]
EQUATIONS = 14          # per prompt; the distractor chain fills EQUATIONS - 2 * d
PAIRS_PER_DEPTH = 10
DEV_SEED = 3001         # the main grid uses a different seed
BUDGETS = [4000, 8000, 16000]
SAMPLING_SEED = 42
MODULUS, MAX_FACTOR, MAX_OFFSET = 97, 5, 19   # each update is (factor * x + offset) mod MODULUS

# Harder arithmetic per step, single agent only:
# RUN_NAME, DEPTHS, BUDGETS, MODULUS, MAX_FACTOR, MAX_OFFSET = "calibrate_hard", [2, 6], [8000, 16000], 997, 99, 199

# Development run of the split systems, for example:
# RUN_NAME, SYSTEMS, DEPTHS, PAIRS_PER_DEPTH, BUDGETS = "dev_systems", ["ensemble", "hierarchy"], [1, 6], 5, [16000]

assert all(1 <= 2 * d < EQUATIONS for d in DEPTHS), "each prompt needs at least one distractor update"

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
OUT = f"/content/drive/MyDrive/small-multi-agents/results/{RUN_NAME}"
!mkdir -p {OUT}

In [ ]:
REPO = "/content/small-multi-agents"
!test -d {REPO} || git clone https://github.com/liatzheng/small-multi-agents.git {REPO}
%cd {REPO}
!git pull --ff-only && git log -1 --format='%H %s'

vLLM 0.33.0 brings its own matching torch, torchvision, and transformers, replacing Colab's. The runner runs as a subprocess, so no runtime restart is needed.

In [ ]:
!pip install -q -r requirements-vllm.txt
!python -c "import vllm, torch, transformers; print('vllm', vllm.__version__, 'torch', torch.__version__, 'transformers', transformers.__version__, torch.cuda.get_device_name())" | tee {OUT}/environment.txt
!git rev-parse HEAD | tee -a {OUT}/environment.txt

## Problems

Generation is deterministic, so rerunning this cell rewrites the same file and resuming stays valid.

In [ ]:
PROBLEMS = f"{OUT}/problems.jsonl"
!rm -f {PROBLEMS}
for d in DEPTHS:
    distractor = EQUATIONS - 2 * d
    !python generate_problems.py --pairs {PAIRS_PER_DEPTH} --seed {DEV_SEED} --chain-length {d} --distractor-length {distractor} --modulus {MODULUS} --max-factor {MAX_FACTOR} --max-offset {MAX_OFFSET} >> {PROBLEMS}
!wc -l {PROBLEMS}

## Run

Every system in `SYSTEMS` at every budget, loading the model once. Everything the runner prints is also appended to `run.log`.

In [ ]:
from notebook_tools import run_logged

run_logged(["python", "run_vllm.py", PROBLEMS, "--out-dir", OUT, "--systems", *SYSTEMS,
            "--budgets", *map(str, BUDGETS), "--seed", str(SAMPLING_SEED), "--resume"], f"{OUT}/run.log")

## Results

Accuracy by system, depth, problem kind, and budget. `length_limit` is the share of problems where the model ran out of tokens before answering; for the ensemble it counts problems where any attempt ran out.

In [ ]:
import json
from pathlib import Path

import pandas as pd

rows = []
hierarchy_calls = []
for path in sorted(Path(OUT).glob("b*/*.jsonl")):
    for line in path.read_text().splitlines():
        r = json.loads(line)
        ran_out = r["status"] == "length_limit" or any(
            a["status"] == "length_limit" for a in r.get("attempts", []))
        rows.append({
            "system": path.stem, "d": r["chain_length"], "kind": r["kind"], "budget": r["budget"],
            "pair_id": r["pair_id"], "status": r["status"], "correct": r["correct"],
            "ran_out": ran_out, "failed_stage": r.get("failed_stage") or "merge",
            "output": r["output_tokens"], "total": r["total_tokens"],
        })
        for call in r.get("calls", []):
            hierarchy_calls.append({
                "d": r["chain_length"], "kind": r["kind"], "budget": r["budget"], "stage": call["stage"],
                "stop_reason": call["stop_reason"], "cap": call["budget"], "output": call["output_tokens"],
            })
df = pd.DataFrame(rows)
df.to_csv(f"{OUT}/summary_problems.csv", index=False)

by_depth = df.groupby(["system", "d", "kind", "budget"]).agg(
    accuracy=("correct", "mean"),
    problems=("correct", "size"),
    length_limit=("ran_out", "mean"),
    wrong_answer=("status", lambda s: (s == "wrong_answer").mean()),
    mean_output=("output", "mean"),
).round(2)
by_depth.to_csv(f"{OUT}/summary_by_depth.csv")
by_depth

In [ ]:
# Accuracy grid: rows are system, depth, and kind; columns are budgets.
by_depth["accuracy"].unstack("budget")

Hierarchy only: how each stage spends its tokens. `ran_out` is the share of calls at that stage that hit their cap. The second table counts unsolved hierarchy problems by the stage where they stopped (`merge` with `wrong_answer` means all four stages finished).

In [ ]:
if hierarchy_calls:
    calls = pd.DataFrame(hierarchy_calls)
    stages = calls.groupby(["budget", "stage"], sort=False).agg(
        calls=("output", "size"),
        mean_cap=("cap", "mean"),
        mean_output=("output", "mean"),
        max_output=("output", "max"),
        ran_out=("stop_reason", lambda s: (s != "stop").mean()),
    ).round(2)
    failed = df[(df.system == "hierarchy") & ~df.correct].groupby(
        ["budget", "failed_stage", "status"]).size().rename("problems")
    stages.to_csv(f"{OUT}/summary_hierarchy_stages.csv")
    display(stages)
    display(failed)
else:
    print("No hierarchy runs in this folder.")

## Reading the results

- Pick the middle budget where the single agent usually finishes (`length_limit` low) but is not near 100% at the larger depths.
- Keep depths whose accuracy at that budget sits roughly between 20% and 80% for at least one kind. Drop depths that are saturated or near zero at every budget.
- If every depth is near 100% once the model has enough tokens, make each step harder with `MODULUS`, `MAX_FACTOR`, and `MAX_OFFSET` (see the `calibrate_hard` line in the settings cell). Raising `EQUATIONS` mostly adds reading, which costs tokens without adding dependency.
- Count `wrong_answer` separately from `length_limit`: wrong answers show the model making mistakes, which voting and decomposition could fix; running out of tokens only shows the budget is too small.

With 10 pairs per depth, each problem is 10 percentage points within a kind, so treat this as a coarse calibration.